# 🚀 vLLM 部署与调优实战

**本文目标**：系统掌握 vLLM 的生产部署、性能调优和监控。

读完这篇你会理解：Docker/K8s 部署方案、性能参数调优矩阵、监控指标体系、常见问题排查。

## 1. 部署方案

### 1.1 Docker 部署

```dockerfile
FROM nvidia/cuda:12.1-runtime-ubuntu22.04
RUN pip install vllm
EXPOSE 8000
ENTRYPOINT ["vllm", "serve", "--host", "0.0.0.0", "--port", "8000"]
```

```bash
# 启动 (单 GPU)
docker run --gpus all -p 8000:8000 \
  -v /path/to/models:/models \
  vllm/vllm-openai:latest \
  --model /models/Llama-3-8B-Instruct

# Docker Compose 核心配置
services:
  vllm:
    image: vllm/vllm-openai:latest
    runtime: nvidia
    ports: ["8000:8000"]
    volumes:
      - /data/models:/models
    command: >
      --model /models/Llama-3-8B-Instruct
      --max-model-len 8192
      --gpu-memory-utilization 0.90
```

### 1.2 生产级架构

```
              ┌──────────────┐
              │  Load Balancer│
              └──┬──┬──┬─────┘
                 │  │  │
     ┌───────────┘  │  └───────────┐
     ▼              ▼              ▼
┌──────────┐  ┌──────────┐  ┌──────────┐
│ vLLM #1  │  │ vLLM #2  │  │ vLLM #3  │
│ model-A  │  │ model-A  │  │ model-B  │
│ GPU:0    │  │ GPU:1    │  │ GPU:2,3  │
└──────────┘  └──────────┘  └──────────┘
     │              │              │
     └──────────────┴──────────────┘
                    │
         ┌──────────┴──────────┐
         │   Prometheus/Grafana│
         └─────────────────────┘
```

## 2. 性能参数调优矩阵

### 2.1 核心参数

| 参数 | 默认值 | 作用 | 何时调大 | 何时调小 |
|------|--------|------|---------|---------|
| max-num-seqs | 256 | 最大并发请求数 | GPU 显存充裕 | OOM 或延迟高 |
| max-num-batched-tokens | 可配 | 每 batch 最大 token 数 | 长 prefill 场景 | 延迟敏感 |
| gpu-memory-utilization | 0.90 | GPU 显存使用率上限 | 显存充裕 | 需要 KV Cache 空间 |
| max-model-len | 自动 | 最大 context 长度 | Agent 场景 | 节省显存 |
| block-size | 16 | KV Cache block 大小 | 长序列 | 短序列 |
| swap-space | 4 GB | CPU swap 空间 | 高并发/可抢占 | CPU 不足 |

### 2.2 按场景调优

```bash
# 场景 1: 高并发短对话 (ChatGPT-like)
vllm serve model \
    --max-num-seqs 128 \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.92

# 场景 2: Agent 服务 (长 context + prefix caching)
vllm serve model \
    --max-num-seqs 16 \
    --max-model-len 16384 \
    --gpu-memory-utilization 0.88 \
    --enable-prefix-caching

# 场景 3: 多模态服务
vllm serve llava-model \
    --max-num-seqs 8 \
    --limit-mm-per-prompt image=5 \
    --gpu-memory-utilization 0.85 \
    --enforce-eager

# 场景 4: 批量离线推理 (激进配置)
vllm serve model \
    --max-num-seqs 256 \
    --max-model-len 32768 \
    --gpu-memory-utilization 0.95
```

### 2.3 Tensor Parallel 配置

```bash
# 单机多卡 (8B 模型不建议 TP, 70B 建议 TP=4)
vllm serve Llama-3-70B-Instruct \
    --tensor-parallel-size 4

# 多节点 (需要 Ray cluster)
vllm serve model \
    --tensor-parallel-size 8 \
    --pipeline-parallel-size 2
```

通信开销参考：TP=2 约 5% overhead，TP=4 约 10% overhead。只有当单卡装不下时才用 TP。

## 3. 监控指标体系

### 3.1 Prometheus Metrics

```bash
# vLLM 内置 metrics endpoint
curl http://localhost:8000/metrics
```

关键指标速查：

| 指标 | 含义 | 告警阈值 |
|------|------|---------|
| vllm:num_requests_running | 当前 running 请求 | — |
| vllm:num_requests_waiting | 当前排队请求 | > 10 → Warning |
| vllm:num_requests_swapped | 被抢占 (swapped) | > 0 → Warning |
| vllm:gpu_cache_usage_perc | KV Cache 使用率 | > 90% → Warning |
| prefix_cache_hit_rate | APC 命中率 | < 10% → 考虑关闭 APC |
| time_to_first_token_seconds | TTFT 分布 | P99 > 2s → Critical |
| time_per_output_token_seconds | TPOT 分布 | P99 > 100ms → Warning |
| request_e2e_time_seconds | 端到端延迟 | 按 SLA 设定 |

### 3.2 核心 Dashboard 面板

```
面板 1: 请求概览
  - running/waiting/swapped 时间序列
  - requests/second, tokens/second

面板 2: 延迟
  - TTFT P50/P95/P99
  - TPOT P50/P95/P99
  - E2E latency histogram

面板 3: 显存
  - GPU memory used/total
  - KV Cache utilization %
  - Prefix cache hit rate

面板 4: 调度健康
  - Preemption rate
  - Batch size 分布
  - Queue length
```

## 4. 常见问题排查

### 4.1 OOM (显存不足)

```bash
# 症状: CUDA out of memory, 请求被频繁抢占

# 排查:
# 1. 检查 KV Cache 使用率
curl http://localhost:8000/metrics | grep gpu_cache_usage_perc

# 2. 检查 prefix caching 是否过多占用
# 3. 检查 max-model-len 是否设得太大
# 4. 检查是否有长 Agent 会话未释放

# 解决:
--gpu-memory-utilization 0.85   # 降低上限
--max-model-len 8192            # 减小 context
--max-num-seqs 32               # 限制并发
--swap-space 16                 # 增大 CPU swap
```

### 4.2 高延迟

```bash
# 症状: TTFT P99 > 5s

# 排查:
# 1. 是 prefill 慢还是排队慢? → 看 waiting queue 长度
# 2. 长 prefill 是否阻塞短请求? → 看 batch token 分布
# 3. GPU 利用率低? → nvidia-smi dmon 检查

# 解决:
--max-num-batched-tokens 4096   # 拆分长 prefill
--enable-chunked-prefill        # prefill chunking
--schedule-policy priority      # 短请求优先
```

### 4.3 低吞吐

```bash
# 症状: GPU 利用率 < 50%, tokens/s 低

# 排查:
# 1. 并发是否足够? → 增加客户端并发
# 2. Batch 是否太小? → 增大 max-num-seqs
# 3. CUDA graph 是否不兼容? → 检查 --enforce-eager

# 解决: 增加并发 + 确保 CUDA graph 开启
--max-num-seqs 128
```

## 5. 性能测试

```bash
# vLLM 内置 benchmark
python -m vllm.entrypoints.openai.run_batch \
    --model model-name \
    --input-file prompts.jsonl \
    --max-tokens 128

# 第三方工具
pip install vllm-benchmark
vllm-benchmark serve \
    --model model-name \
    --num-prompts 1000 \
    --request-rate 10

# 关键测试场景:
# 1. 纯吞吐: 高并发 (256+), 短 prompt (100 tokens)
# 2. TTFT 测试: 固定 1 并发, 变长 prompt
# 3. TPOT 测试: 变长 decode (100/500/2000 tokens)
# 4. 混合测试: 80% 短 + 20% 长, 观察 P99
```

### 5.1 Benchmark 脚本

```python
import asyncio
import aiohttp
import time

async def benchmark(url, model, num_requests=100, concurrency=10):
    sem = asyncio.Semaphore(concurrency)
    latencies = []

    async def make_request(i):
        async with sem:
            start = time.time()
            async with aiohttp.ClientSession() as session:
                payload = {
                    "model": model,
                    "messages": [{"role": "user",
                        "content": f"Say hello {i}"}],
                    "max_tokens": 50
                }
                async with session.post(
                    f"{url}/v1/chat/completions",
                    json=payload) as resp:
                    await resp.json()
            latencies.append(time.time() - start)

    tasks = [make_request(i) for i in range(num_requests)]
    await asyncio.gather(*tasks)

    latencies.sort()
    n = len(latencies)
    print(f"Requests: {n}, Concurrency: {concurrency}")
    print(f"  P50: {latencies[n//2]*1000:.0f}ms")
    print(f"  P95: {latencies[int(n*0.95)]*1000:.0f}ms")
    print(f"  P99: {latencies[int(n*0.99)]*1000:.0f}ms")
    print(f"  Throughput: {n/max(latencies):.1f} req/s")
```

## 6. 生产 Checklist

部署前确认：

- [ ] 模型下载并验证了 checksum
- [ ] GPU 显存足够 (模型权重 + 预期 KV Cache 峰值)
- [ ] max-model-len 匹配业务需求 (且 GPU 显存能承受)
- [ ] 开启了 prefix caching (Agent 场景)
- [ ] metrics endpoint 可被 Prometheus 访问
- [ ] 设置了合理的健康检查 (readiness probe)
- [ ] 灰度发布: 先 10% 流量, 观察 30min
- [ ] 准备了降级方案: 如果 vLLM 挂了, fallback 到备选

调优流程：

1. 部署基线配置 → 跑 benchmark → 记录基线指标
2. 逐步增加并发 (max-num-seqs) → 找到吞吐拐点
3. 调整 gpu-memory-utilization → 平衡 KV Cache 和稳定性
4. 开启 prefix caching → 观察命中率和额外显存消耗
5. 根据业务场景调整 max-num-batched-tokens
6. 稳定运行 24h → 检查 P99 延迟和 OOM 事件
7. 上线后持续监控 → 根据流量模式定期调整